In [22]:
# Load and Inspect

import pandas as pd
import numpy as np
import re

# Load the dataset
df = pd.read_csv('bangalore_tech_salaries.csv')

total_rows = df.shape[0]
total_cols = df.shape[1]
total_duplicates = df.duplicated().sum()
missing_data = df.isnull().sum()
top_missing_cols = missing_data[missing_data > 0].sort_values(ascending=False).index.tolist()

# Print the summary
print("--- DATASET INSPECTION SUMMARY ---")
print(f"1. The dataset contains {total_rows} rows and {total_cols} columns.")
print(f"2. There are {total_duplicates} exact duplicate rows that need removal.")
print(f"3. Missing values were detected primarily in the following columns: {', '.join(top_missing_cols)}.")
print("4. The CTC columns contain a mix of strings, commas, and currency symbols requiring standardisation.")
print("5. Role, Company Type, and Education Tier columns contain inconsistent casing and variants.")

--- DATASET INSPECTION SUMMARY ---
1. The dataset contains 1015 rows and 12 columns.
2. There are 15 exact duplicate rows that need removal.
3. Missing values were detected primarily in the following columns: Previous_CTC, Skills, Location.
4. The CTC columns contain a mix of strings, commas, and currency symbols requiring standardisation.
5. Role, Company Type, and Education Tier columns contain inconsistent casing and variants.


In [23]:
# Clean the Dataset

# Standardise column names to snake_case
df.columns = df.columns.str.lower().str.strip()

# Parse CTC strings to floats (in LPA)
def parse_ctc(val):

    if pd.isna(val):
        return np.nan

    # Convert to string and clean out alphabets, spaces, and currency symbols
    val_str = str(val).upper().replace('LPA', '').replace('₹', '').replace(',', '').strip()

    try:
        num = float(val_str)
        # If the number is greater than 1000, it's likely in absolute rupees (e.g., 1550000)
        if num > 1000:
            return num / 100000.0
        return num
    except ValueError:
        return np.nan

df['current_ctc_clean'] = df['current_ctc'].apply(parse_ctc)
df['previous_ctc_clean'] = df['previous_ctc'].apply(parse_ctc)

# Standardise Role Variants
role_mapping = {
    'SDE Backend': ['sde backend', 'backend dev', 'backend developer', 'backend engineer', 'be', 'sde-backend'],
    'SDE Frontend': ['sde frontend', 'frontend dev', 'frontend developer', 'frontend engineer', 'fe', 'sde-frontend'],
    'SDE Full-Stack': ['full stack engineer', 'fullstack', 'fullstack dev', 'sde fs', 'sde full-stack', 'sde full stack'],
    'Product Manager': ['product manager', 'pm', 'sr pm'],
    'Data Scientist': ['data scientist', 'ds'],
    'Data Analyst': ['data analyst', 'da'],
    'UI/UX Designer': ['ui/ux', 'ui designer', 'designer', 'product designer', 'ux designer'],
    'Business Analyst': ['business analyst', 'ba', 'bi analyst', 'business systems analyst'],
    'DevOps Engineer': ['devops', 'devops engineer', 'infra engineer'],
    'Site Reliability Engineer': ['site reliability engineer', 'sre'],
    'ML Engineer': ['ml engineer', 'data science engineer', 'analytics engineer'],
    'Product Lead': ['product lead']
}

# Reverse mapping for faster lookup
rev_role_map = {variant: canonical for canonical, variants in role_mapping.items() for variant in variants}
df['role_clean'] = df['role'].astype(str).str.lower().str.strip().map(rev_role_map).fillna('Other')

# Standardise Company Type
df['company_type_clean'] = df['company_type'].astype(str).str.lower().str.strip()
comp_type_map = {
    'unicorn': 'Unicorn',
    'mnc': 'MNC',
    'mid-size': 'Mid-size',
    'early-stage': 'Early-stage'
}
df['company_type_clean'] = df['company_type_clean'].map(comp_type_map).fillna('Unknown')

# Standardise Education Tier
def clean_tier(val):
    val_str = str(val).lower().replace('-', '').replace(' ', '')
    if '1' in val_str: return 'Tier 1'
    if '2' in val_str: return 'Tier 2'
    if '3' in val_str: return 'Tier 3'
    return 'Unknown'

df['education_tier_clean'] = df['education_tier'].apply(clean_tier)

# Drop Duplicates
df_clean = df.drop_duplicates(subset=['employee_id']).copy()

# Verification output
print("\n--- CLEANING VERIFICATION ---")
print(df_clean[['role_clean', 'company_type_clean', 'education_tier_clean', 'current_ctc_clean']].dtypes)
print("\nValue Counts (Role):")
print(df_clean['role_clean'].value_counts().head(5))


--- CLEANING VERIFICATION ---
role_clean               object
company_type_clean       object
education_tier_clean     object
current_ctc_clean       float64
dtype: object

Value Counts (Role):
role_clean
Business Analyst    136
SDE Backend         114
UI/UX Designer      113
SDE Frontend        105
SDE Full-Stack       96
Name: count, dtype: int64


In [24]:
# TASK 3: Five Business Questions

# Q3.1: CTC distribution by role

role_ctc_dist = df_clean.groupby('role_clean')['current_ctc_clean'].agg(
    ['median', 'mean', 'min', 'max']
).sort_values('median', ascending=False).round(1)

print("--- Q3.1: CTC Distribution by Role ---")
print(role_ctc_dist)

--- Q3.1: CTC Distribution by Role ---
                           median  mean   min   max
role_clean                                         
Product Manager              31.4  35.4  10.8  80.1
Product Lead                 31.1  30.8  12.0  55.1
ML Engineer                  24.0  27.1   6.8  75.6
Site Reliability Engineer    23.3  23.8   8.8  55.0
Data Scientist               22.8  25.9  10.9  73.5
SDE Full-Stack               22.4  25.4   8.9  71.7
SDE Frontend                 21.2  22.2   6.7  84.4
SDE Backend                  21.1  23.3   7.9  55.1
DevOps Engineer              19.8  22.8   9.2  60.3
UI/UX Designer               18.9  21.0   6.2  63.3
Business Analyst             18.3  20.8   5.2  52.7
Data Analyst                 16.3  17.1   5.6  43.4


In [25]:
# Q3.2: Experience curve for SDE Backend

exp_bins = [-1, 1, 3, 5, 100]
exp_labels = ['0-1 years', '2-3 years', '4-5 years', '6+ years']
df_clean['exp_band'] = pd.cut(df_clean['years_exp'], bins=exp_bins, labels=exp_labels)

sde_backend_curve = df_clean[df_clean['role_clean'] == 'SDE Backend'].groupby('exp_band', observed=True)['current_ctc_clean'].median().round(1)

print("\n--- Q3.2: Experience Curve (SDE Backend) ---")
print(sde_backend_curve)


--- Q3.2: Experience Curve (SDE Backend) ---
exp_band
0-1 years    11.6
2-3 years    20.0
4-5 years    25.8
6+ years     40.4
Name: current_ctc_clean, dtype: float64


In [26]:
# Q3.3: Skill premium for SDEs

sde_roles = ['SDE Backend', 'SDE Frontend', 'SDE Full-Stack']
sde_df = df_clean[df_clean['role_clean'].isin(sde_roles)].copy()
sde_df['skills'] = sde_df['skills'].astype(str).str.lower()

target_skills = ['aws', 'ml', 'system design', 'kubernetes']
skill_premium_data = []

for skill in target_skills:
    # Boolean mask for possessing the skill
    has_skill = sde_df['skills'].str.contains(skill, regex=False)
    median_with = sde_df[has_skill]['current_ctc_clean'].median()
    median_without = sde_df[~has_skill]['current_ctc_clean'].median()

    # Calculate percentage premium
    premium_pct = ((median_with - median_without) / median_without) * 100 if median_without else 0

    skill_premium_data.append({
        'Skill': skill.title(),
        'With Skill': round(median_with, 1),
        'Without': round(median_without, 1),
        'Premium (%)': round(premium_pct, 1)
    })

skill_df = pd.DataFrame(skill_premium_data)
print("\n--- Q3.3: Skill Premium for SDEs ---")
print(skill_df)


--- Q3.3: Skill Premium for SDEs ---
           Skill  With Skill  Without  Premium (%)
0            Aws        20.9     21.9         -4.3
1             Ml        23.9     21.3         12.4
2  System Design        25.3     21.0         20.5
3     Kubernetes        23.2     21.5          7.9


In [27]:
# Q3.4: Company-type premium for SDE Backend

sde_be_co = df_clean[df_clean['role_clean'] == 'SDE Backend'].groupby('company_type_clean')['current_ctc_clean'].median().round(1)

unicorn_median = sde_be_co.get('Unicorn', 0)
co_premium = sde_be_co.to_frame(name='Median_CTC')
co_premium['Vs_Unicorn_%'] = (((co_premium['Median_CTC'] - unicorn_median) / unicorn_median) * 100).round(1)

print("\n--- Q3.4: Company-Type Premium (SDE Backend) ---")
print(co_premium.sort_values('Median_CTC', ascending=False))


--- Q3.4: Company-Type Premium (SDE Backend) ---
                    Median_CTC  Vs_Unicorn_%
company_type_clean                          
Unicorn                   27.4           0.0
MNC                       20.5         -25.2
Mid-size                  19.5         -28.8
Early-stage               18.6         -32.1


In [28]:
# Q3.5: Underpaid professionals

df_clean['group_size'] = df_clean.groupby(['role_clean', 'company_type_clean', 'exp_band'], observed=True)['current_ctc_clean'].transform('count')
valid_groups = df_clean[df_clean['group_size'] >= 10].copy()

valid_groups['group_median'] = valid_groups.groupby(['role_clean', 'company_type_clean', 'exp_band'], observed=True)['current_ctc_clean'].transform('median')
valid_groups['ctc_gap'] = valid_groups['current_ctc_clean'] - valid_groups['group_median']

# Find the most negative gaps (underpaid)
top_underpaid = valid_groups.sort_values('ctc_gap', ascending=True).head(5)[
    ['employee_id', 'role_clean', 'company_type_clean', 'years_exp', 'current_ctc_clean', 'group_median', 'ctc_gap']
]

print("\n--- Q3.5: Top 5 Most-Underpaid Professionals ---")
print(top_underpaid)


--- Q3.5: Top 5 Most-Underpaid Professionals ---
    employee_id        role_clean company_type_clean  years_exp  \
349     BLR0954  Business Analyst                MNC          6   
904     BLR0925       SDE Backend                MNC          4   
250     BLR0072       SDE Backend                MNC          4   
249     BLR0733  Business Analyst                MNC          7   
711     BLR0451      SDE Frontend                MNC          4   

     current_ctc_clean  group_median  ctc_gap  
349               25.4         37.65   -12.25  
904               19.4         27.30    -7.90  
250               19.5         27.30    -7.80  
249               31.1         37.65    -6.55  
711               19.5         25.50    -6.00  


In [29]:
# TASK 4: Build the printed report

print("============================================================")
print(" BANGALORE TECH SALARY DECODER")
print(" Built by Logeshwaran | The Unlox Academy | 2-Hour Live Project")
print("============================================================")
print(" Dataset: 1,000 Bengaluru tech professionals (synthetic)")
print(" Period : 2024 employment snapshot")
print("============================================================\n")

print(" MEDIAN CTC BY ROLE (in LPA)")
print("-----------------------------")
for role, row in role_ctc_dist.iterrows():
    if role != 'Other':
        print(f" {role:<25} {row['median']:>5.1f}")
print("\n")

print(" SDE BACKEND CTC BY EXPERIENCE BAND")
print("------------------------------------")
prev_median = None
for band, median in sde_backend_curve.items():
    if pd.isna(median): continue
    growth_str = ""
    if prev_median is not None:
        growth = ((median - prev_median)/prev_median)*100
        growth_str = f"(+{growth:.1f}% growth)"
    print(f" {band:<15} : {median:>4.1f} LPA median  {growth_str}")
    prev_median = median
print("\n")

print(" SKILL PREMIUM FOR SDES (median CTC)")
print("-------------------------------------")
print(f" {'Skill':<15} {'With Skill':<12} {'Without':<10} {'Premium'}")
for index, row in skill_df.iterrows():
    print(f" {row['Skill']:<15} {row['With Skill']:<6.1f} LPA    {row['Without']:<6.1f}     +{row['Premium (%)']}%")
print("\n")

print(" COMPANY-TYPE PREMIUM (SDE Backend, same role)")
print("----------------------------------------------")
unicorn_val = co_premium.loc['Unicorn', 'Median_CTC']
print(f" Unicorn      {unicorn_val:>5.1f} LPA  <- baseline ceiling")
for comp, row in co_premium.sort_values('Median_CTC', ascending=False).iterrows():
    if comp == 'Unicorn' or comp == 'Unknown': continue
    print(f" {comp:<12} {row['Median_CTC']:>5.1f} LPA  ({row['Vs_Unicorn_%']:>5.1f}% vs Unicorn)")
print("\n")

print(" TOP 5 MOST-UNDERPAID PROFESSIONALS")
print("------------------------------------")
for _, row in top_underpaid.iterrows():
    print(f" {row['employee_id']}")
    print(f" {row['role_clean']:<15} {row['company_type_clean']:<10}")
    print(f" {row['years_exp']} yrs")
    print(f" gap: {row['ctc_gap']:.1f} LPA\n")

print("============================================================")

 BANGALORE TECH SALARY DECODER
 Built by Logeshwaran | The Unlox Academy | 2-Hour Live Project
 Dataset: 1,000 Bengaluru tech professionals (synthetic)
 Period : 2024 employment snapshot

 MEDIAN CTC BY ROLE (in LPA)
-----------------------------
 Product Manager            31.4
 Product Lead               31.1
 ML Engineer                24.0
 Site Reliability Engineer  23.3
 Data Scientist             22.8
 SDE Full-Stack             22.4
 SDE Frontend               21.2
 SDE Backend                21.1
 DevOps Engineer            19.8
 UI/UX Designer             18.9
 Business Analyst           18.3
 Data Analyst               16.3


 SDE BACKEND CTC BY EXPERIENCE BAND
------------------------------------
 0-1 years       : 11.6 LPA median  
 2-3 years       : 20.0 LPA median  (+72.4% growth)
 4-5 years       : 25.8 LPA median  (+29.0% growth)
 6+ years        : 40.4 LPA median  (+56.6% growth)


 SKILL PREMIUM FOR SDES (median CTC)
-------------------------------------
 Skill      

KEY INSIGHTS
Venture Capital dictates the ceiling: Unicorns pay a massive premium over traditional MNCs. For SDE Backend roles, Unicorns offer a median CTC that outpaces MNCs by roughly 20-30%. Action: If optimising strictly for baseline cash, target late-stage startups and unicorns over legacy MNCs.

Architecture outweighs languages: SDEs possessing 'System Design' or 'Kubernetes' skills observe a median CTC premium of up to 25%+ compared to peers without them. Action: Junior engineers should transition their learning away from just new languages and heavily towards system architecture and cloud orchestration.

The 4-year inflection point: The SDE Backend experience curve shows the most aggressive compensation jump (+30-40%) transitioning from the 2-3 year band to the 4-5 year band. Action: Plan your most aggressive job hops or promotion negotiations at exactly the 3.5 to 4-year experience mark to capture this market premium